In [21]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from typing import Optional, Union

In [2]:
sep = "\t"
chunksize = 1000000
data_path = "./criteo/train.txt"

In [ ]:
def chunk_load(
    path: str,
    sample_ratio: float,
    seed: int,
    chunksize: Optional[int] = None,
    sep: Optional[str] = None,
) -> pd.DataFrame:
    """Load data in chunks and sample from each chunk."""
    data_chunk = pd.read_csv(path, chunksize=chunksize, sep=sep, header=None)
    data_temp = []
    for chunk in tqdm(data_chunk):
        sample_chunk = chunk.sample(frac=sample_ratio, random_state=seed)
        data_temp.append(sample_chunk)
    data = pd.concat(data_temp, axis=0)

    # 获取总列数
    n_cols = data.shape[1]
    # 创建列名列表
    column_names = ["Label"]  # 第一列
    column_names.extend(
        [f"C_{i}" for i in range(1, 14)]
    )  # C_1 到 C_13（第2-14列，共13列）
    # 剩余的列用 I_1, I_2, ... 命名
    remaining_cols = n_cols - 14  # 减去已命名的14列
    if remaining_cols > 0:
        column_names.extend([f"I_{i}" for i in range(1, remaining_cols + 1)])
    # 应用列名
    data.columns = column_names
    del data_temp, data_chunk, sample_chunk, chunk
    return data

In [6]:
train_data = chunk_load(data_path, 0.1, 42, chunksize=chunksize, sep=sep)
train_data.head()

46it [04:36,  6.02s/it]


,Label,C_1,C_2,C_3,C_4,C_5,C_6,C_7,C_8,C_9,...,I_17,I_18,I_19,I_20,I_21,I_22,I_23,I_24,I_25,I_26
987231,0,0.0,5,5.0,1.0,8680.0,NaN,NaN,15.0,NaN,...,e5ba7672,698d1c68,NaN,NaN,15fce809,NaN,dbb486d7,f96a556f,NaN,NaN
79954,0,0.0,7,7.0,1.0,2060.0,63.0,24.0,16.0,225.0,...,e5ba7672,3182300e,55dd3565,b1252a9d,be87db85,NaN,3a171ecb,45ab94c8,010f6491,c84c4aec
567130,0,0.0,0,NaN,12.0,21133.0,616.0,1.0,25.0,272.0,...,d4bb7bd8,c4ba79ab,NaN,NaN,69c7ea42,NaN,32c7478e,8fc66e78,NaN,NaN
500891,0,0.0,4,5.0,3.0,2088.0,NaN,0.0,3.0,2.0,...,e5ba7672,005c6740,21ddcdc9,5840adea,2ce86272,NaN,32c7478e,1793a828,e8b83407,b9809574
55399,0,NaN,2,1.0,5.0,311411.0,NaN,0.0,7.0,29.0,...,1e88c74f,52e44668,NaN,NaN,e587c466,NaN,bcdee96c,3b183c5c,NaN,NaN


In [7]:
train_data.shape

(4584062, 40)

In [8]:
from sklearn.preprocessing import LabelEncoder


def preprocess_data(df: pd.DataFrame) -> Union[pd.DataFrame, dict]:
    """Preprocess data."""
    # 均值填充
    df.iloc[:, 1:14] = df.iloc[:, 1:14].fillna(df.iloc[:, 1:14].mean())
    # 中位数填充
    # df.iloc[:, 1:14] = df.iloc[:, 1:14].fillna(df.iloc[:, 1:14].median())
    # 前向填充（ffill）或后向填充（bfill）
    # df.iloc[:, 1:14] = df.iloc[:, 1:14].ffill().bfill()

    # 离散特征：众数填充
    for col in range(14, 40):
        df.iloc[:, col] = df.iloc[:, col].fillna(df.iloc[:, col].mode()[0])

    # Label Encoding 离散特征
    label_encoders = {}
    for col in range(14, 40):
        le = LabelEncoder()
        df.iloc[:, col] = le.fit_transform(df.iloc[:, col])
        label_encoders[col] = le  # 保存编码器以便逆编码或一致性转换
    return df, label_encoders

In [9]:
trans_train_data, label_encoders = preprocess_data(train_data)

In [ ]:
import pickle


def save_df_parquet(dataframe, label_encoders, filename_base="criteo_data"):
    # 存储DataFrame为parquet
    dataframe.to_parquet(f"{filename_base}.parquet")

    # 存储编码器到pickle
    with open(f"{filename_base}_encoders.pkl", "wb") as f:
        pickle.dump(label_encoders, f)

In [48]:
save_df_parquet(trans_train_data, label_encoders)
# loaded_df, loaded_encoders = load_data_combined()